## PartA: Reproject layers and compute community area size

In [3]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [4]:
# community_dir = Path("/home/metayj/InternetGIS/CrimeAnalysis/data/processed")
community_gdf = gpd.read_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/community_areas.geojson")
crime_gdf = gpd.read_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/crime_points_2025.geojson")
cta_gdf = gpd.read_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/cta_stations.geojson")
police_gdf = gpd.read_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/police_stations.geojson")
biz_gdf = gpd.read_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/business_licenses_clean.geojson")

# Reproject to a projected CRS for area/distance calculations
target_crs = "EPSG:26971"

community_proj = community_gdf.to_crs(target_crs).copy()
crime_proj = crime_gdf.to_crs(target_crs).copy()
cta_proj = cta_gdf.to_crs(target_crs).copy()
police_proj = police_gdf.to_crs(target_crs).copy()
biz_proj = biz_gdf.to_crs(target_crs).copy()

# Compute area in square kilometers
community_proj["area_sqkm"] = community_proj.geometry.area / 1_000_000

# Keep essential fields only
community_proj = community_proj[["AREA_NUMBE", "COMMUNITY", "area_sqkm", "geometry"]]

print(community_proj.head())
print(community_proj.crs)

   AREA_NUMBE       COMMUNITY  area_sqkm  \
0           1     ROGERS PARK   4.762220   
1           2      WEST RIDGE   9.144399   
2           3          UPTOWN   6.047607   
3           4  LINCOLN SQUARE   6.628875   
4           5    NORTH CENTER   5.300527   

                                            geometry  
0  MULTIPOLYGON (((356237.576 592121.821, 356139....  
1  MULTIPOLYGON (((353726.032 594470.382, 353727....  
2  MULTIPOLYGON (((357397.618 587314.391, 357151....  
3  MULTIPOLYGON (((354611.686 589658.442, 354612....  
4  MULTIPOLYGON (((354735.595 584798.593, 354731....  
EPSG:26971


## PartB: Spatially join crimes to community areas and count

In [5]:
# Spatial join crimes to community areas
crime_join = gpd.sjoin(
    crime_proj,
    community_proj[["AREA_NUMBE", "COMMUNITY", "geometry"]],
    how="left",
    predicate="within"
)

crime_counts = (
    crime_join.groupby("AREA_NUMBE")
    .size()
    .reset_index(name="crime_count_2025")
)

print(crime_counts.head())
print("Crime records joined:", len(crime_join))
print("Crime records with matched community:", crime_join["AREA_NUMBE"].notna().sum())

   AREA_NUMBE  crime_count_2025
0         1.0              3582
1         2.0              3087
2         3.0              4209
3         4.0              2052
4         5.0              1254
Crime records joined: 236027
Crime records with matched community: 235286


## PartC: Count CTA stations by community area

In [6]:
cta_join = gpd.sjoin(
    cta_proj,
    community_proj[["AREA_NUMBE", "geometry"]],
    how="left",
    predicate="within"
)

cta_counts = (
    cta_join.groupby("AREA_NUMBE")
    .size()
    .reset_index(name="cta_count")
)

print(cta_counts.head())

   AREA_NUMBE  cta_count
0         1.0          4
1         3.0          3
2         4.0          4
3         5.0          2
4         6.0          7


## PartD: Count police stations by community area

In [7]:
police_join = gpd.sjoin(
    police_proj,
    community_proj[["AREA_NUMBE", "geometry"]],
    how="left",
    predicate="within"
)

police_counts = (
    police_join.groupby("AREA_NUMBE")
    .size()
    .reset_index(name="police_count")
)

print(police_counts.head())

   AREA_NUMBE  police_count
0           1             1
1           4             1
2           6             1
3           8             1
4          11             1


## PartE: Count businesses by community area

In [8]:
biz_join = gpd.sjoin(
    biz_proj,
    community_proj[["AREA_NUMBE", "geometry"]],
    how="left",
    predicate="within"
)

biz_counts = (
    biz_join.groupby("AREA_NUMBE")
    .size()
    .reset_index(name="business_count")
)

print(biz_counts.head())
print("Business records joined:", len(biz_join))
print("Business records with matched community:", biz_join["AREA_NUMBE"].notna().sum())

   AREA_NUMBE  business_count
0         1.0           15204
1         2.0           21427
2         3.0           18784
3         4.0           16952
4         5.0           17565
Business records joined: 1099208
Business records with matched community: 1097615


## PartF: Merge all counts into one polygon layer

In [9]:
community_analysis = community_proj.merge(crime_counts, on="AREA_NUMBE", how="left")
community_analysis = community_analysis.merge(cta_counts, on="AREA_NUMBE", how="left")
community_analysis = community_analysis.merge(police_counts, on="AREA_NUMBE", how="left")
community_analysis = community_analysis.merge(biz_counts, on="AREA_NUMBE", how="left")

# Fill missing counts with zero
for col in ["crime_count_2025", "cta_count", "police_count", "business_count"]:
    community_analysis[col] = community_analysis[col].fillna(0)

# Derived density variables
community_analysis["crime_density_sqkm"] = (
    community_analysis["crime_count_2025"] / community_analysis["area_sqkm"]
)

community_analysis["business_density_sqkm"] = (
    community_analysis["business_count"] / community_analysis["area_sqkm"]
)

print(community_analysis.head())

   AREA_NUMBE       COMMUNITY  area_sqkm  \
0           1     ROGERS PARK   4.762220   
1           2      WEST RIDGE   9.144399   
2           3          UPTOWN   6.047607   
3           4  LINCOLN SQUARE   6.628875   
4           5    NORTH CENTER   5.300527   

                                            geometry  crime_count_2025  \
0  MULTIPOLYGON (((356237.576 592121.821, 356139....              3582   
1  MULTIPOLYGON (((353726.032 594470.382, 353727....              3087   
2  MULTIPOLYGON (((357397.618 587314.391, 357151....              4209   
3  MULTIPOLYGON (((354611.686 589658.442, 354612....              2052   
4  MULTIPOLYGON (((354735.595 584798.593, 354731....              1254   

   cta_count  police_count  business_count  crime_density_sqkm  \
0        4.0           1.0           15204          752.170235   
1        0.0           0.0           21427          337.583705   
2        3.0           0.0           18784          695.977731   
3        4.0           1.0

In [10]:
community_analysis.to_file(
    "/home/metayj/InternetGIS/CrimeAnalysis/data/processed/community_area_analysis_v1.geojson",
    driver="GeoJSON"
)

## Inspect somethings after run above code
1. Are there any unmatched points?
Look at how many crimes or businesses did not match a community area.

A small number is okay. A large number would mean boundary mismatch or coordinate issues.

In [11]:
print("Unmatched crimes:", crime_join["AREA_NUMBE"].isna().sum())
print("Unmatched businesses:", biz_join["AREA_NUMBE"].isna().sum())
print("Unmatched CTA stations:", cta_join["AREA_NUMBE"].isna().sum())
print("Unmatched police stations:", police_join["AREA_NUMBE"].isna().sum())

Unmatched crimes: 741
Unmatched businesses: 1593
Unmatched CTA stations: 20
Unmatched police stations: 0


2. Which community areas have the highest crime counts?
You can quickly inspect the top 10

In [12]:
community_analysis.sort_values("crime_count_2025", ascending=False)[
    ["COMMUNITY", "crime_count_2025", "crime_density_sqkm", "business_count", "cta_count", "police_count"]
].head(10)

,COMMUNITY,crime_count_2025,crime_density_sqkm,business_count,cta_count,police_count
24,AUSTIN,11621,627.775477,28300,5.0,1.0
7,NEAR NORTH SIDE,11136,1563.286942,95471,7.0,1.0
27,NEAR WEST SIDE,10275,697.816998,53629,10.0,1.0
31,LOOP,8535,1982.701776,90826,16.0,0.0
42,SOUTH SHORE,8009,1053.721445,11493,0.0,0.0
23,WEST TOWN,6868,579.527734,49300,4.0,0.0
28,NORTH LAWNDALE,6319,760.071933,9778,4.0,1.0
22,HUMBOLDT PARK,6155,659.345399,19389,0.0,0.0
70,AUBURN GRESHAM,6099,624.838016,11108,0.0,1.0
68,GREATER GRAND CROSSING,5947,647.553545,11051,1.0,0.0


In [13]:
# Save current finished layers
community_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/vector/community_areas.gpkg", driver="GPKG")
crime_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/vector/crime_points_2025.gpkg", driver="GPKG")
cta_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/vector/cta_stations.gpkg", driver="GPKG")
police_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/vector/police_stations.gpkg", driver="GPKG")
biz_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/vector/business_licenses_clean.gpkg", driver="GPKG")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
